# Aphasia Recovery Forecasting: 1-Month Follow-Up Analysis
### Objective
This analysis aims to predict the naming success of specific words during the **1-month follow-up phase**. By identifying which factors drive long-term maintenance, we can better understand the recovery process.

## 🗺️ Model Architecture & Flow
This section breaks down how the prediction works for use in explaining the model to others.

### 1. The Inputs (What the model knows)
| Category | Feature Name | Description |
| :--- | :--- | :--- |
| **Mastery** | `s` | Cumulative number of times the patient previously named the word correctly. |
| **Automaticity**| `consecutive_success` | Count of **consecutive correct** attempts. High streaks indicate stable learning. |
| **Dynamics** | `learning_velocity` | The recent 'trend' in accuracy (slopes). Is the patient currently improving? |
| **Stability** | `rt_jitter` | Numerical 'jitter' in naming speed. Shows if the retrieval is stable or hesitant. |
| **Effort** | `rt_efficiency` | **RT per Complexity Point**. Measures if hard words are retrieved with low effort. |
| **Difficulty** | `complexity_score` | Linguistic complexity of the word (length, phonological depth). |
| **Clinical** | `demo.mpo` | Months Post Onset (Time since the stroke). |
| **Patient** | `demo.age` | Patient's age (neurological plasticity proxy). |
| **Engagement**| `stars` | Cumulative gamification rewards (motivation proxy). |

--- 

### 2. The Engine: Random Forest
The model acts like a **Committee of 100 Clinical Decision Trees**:
1. Each 'Tree' looks at a slightly different subset of features.
2. They ask 'If/Then' questions (e.g., *'If consecutive success is > 3 AND rt_efficiency is low, will they maintain?'*).
3. The final prediction is a **weighted vote** across all 100 trees.

### 3. The Output (The Forecast)
- **Binary Label**: 1 (Will be Correct) or 0 (Will be Incorrect) at the 1-month mark.
- **Probability**: A score from 0% to 100% confidence in that maintenance success.

## 🔍 Deep Dive: Advanced Prediction Features
We have introduced two new features to capture the 'Mastery' and 'Effort' of the patient.

### A. Success Streak (Consecutive Corrects)
A patient might have 5 successes in 10 days, but if they are `1-0-1-0-1`, the knowledge is fragile. If they are `0-0-0-0-1-1-1-1-1`, they have achieved **stable acquisition**. The streak counts consecutive 1s.

### B. Retrieval Efficiency (RT / Complexity)
Naming a 'hard' word (Complex) in 2 seconds is more impressive than naming an 'easy' word (Simple) in 2 seconds. This ratio highlights patients who have developed **high neurological efficiency** for specific linguistic targets.

## 1. Data Integration & Engineering
Every line of this code is commented to explain the logic of data merging and the math of the new features.

In [ ]:
# Import libraries for data science and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

# Configure plot aesthetics
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)

# LOAD DATASETS FROM LOCAL STORAGE
probes = pd.read_csv('1. 2019-11-12_naming_probes_tidy.csv')
tx_nop = pd.read_csv('2. 2019-10-25_treatment_retrieval_noprime.csv')
tx_p = pd.read_csv('4. 2019-10-25_treatment_retrieval_prime.csv')
outcomes = pd.read_csv('6. 2019-10-3_Outcome_data_tidy.csv')
rewards = pd.read_csv('5. 2019-12-3_coins-stars.csv')

# CLEAN PATIENT IDs FOR RELIABLE MERGING
for d in [probes, tx_nop, tx_p, outcomes, rewards]:
    d['player'] = d['player'].str.lower()

# CONSTRUCT UNIFIED TIMELINE
p_clean = probes[['player', 'date', 'session', 'target', 'category', 'trial_resp.corr.hand', 'vocal.rt', 'complexity_score', 'phase']].rename(columns={'target':'word', 'trial_resp.corr.hand':'success'})
nop_clean = tx_nop[['player', 'date', 'session', 'stim_text', 'category', 'naming1_resp.corr', 'naming1_vocal.rt']].rename(columns={'stim_text':'word', 'naming1_resp.corr':'success', 'naming1_vocal.rt':'vocal.rt'})
p_clean_tx = tx_p[['player', 'date', 'session', 'stim_text', 'category', 'naming2_resp.corr', 'naming2_vocal.rt']].rename(columns={'stim_text':'word', 'naming2_resp.corr':'success', 'naming2_vocal.rt':'vocal.rt'})

df = pd.concat([p_clean, nop_clean, p_clean_tx])
df['date'] = pd.to_datetime(df['date'].str.replace('_', '-'), errors='coerce', format='mixed')
df['phase'] = df['phase'].fillna('treatment')

# MERGE CLINICAL MARKERS (MPO, AGE)
demos = outcomes.groupby('player').agg({'demo.mpo': 'first', 'demo.age': 'first'}).reset_index()
df = df.merge(demos, on='player', how='left')

# MERGE GAMIFICATION REWARDS
reward_agg = rewards.groupby(['player', 'session', 'category']).agg({'stars': 'sum', 'coins': 'sum'}).reset_index()
df = df.merge(reward_agg, on=['player', 'session', 'category'], how='left').fillna({'stars': 0, 'coins': 0})

# MAP COMPLEXITY TO ALL DATA POINTS
comp_map = probes.groupby('target')['complexity_score'].mean()
df['complexity_score'] = df['complexity_score'].fillna(df['word'].map(comp_map))

# --- ADVANCED CLINICAL FEATURE ENGINEERING ---

# 1. Learning Velocity: Trend of improvement over 3 trials
df['rolling_acc'] = df.groupby(['player', 'word'])['success'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['learning_velocity'] = df.groupby(['player', 'word'])['rolling_acc'].diff().fillna(0)

# 2. RT Jitter: Modeling retrieval instability
df['rt_jitter'] = df.groupby(['player', 'word'])['vocal.rt'].transform(lambda x: x.rolling(window=5, min_periods=1).std() / (x.rolling(window=5, min_periods=1).mean() + 1e-9)).fillna(0)

# 3. SUCCESS STREAK: Count of consecutive 1s (Measures Automaticity)
def get_streak(x):
    # This function resets the count every time a '0' is encountered
    y = x.cumsum()
    return y.sub(y.mask(x != 0).ffill().fillna(0))
df['consecutive_success'] = df.groupby(['player', 'word'])['success'].transform(get_streak).shift(1).fillna(0)

# 4. RETRIEVAL EFFICIENCY: Response time normalized by complexity
df['rt_efficiency'] = df['vocal.rt'] / (df['complexity_score'] + 1e-9)

# 5. Cumulative Success (s)
df['s'] = df.groupby(['player', 'word'])['success'].shift(1).fillna(0).groupby([df['player'], df['word']]).cumsum()

# DROP TRIALS WITH MISSING CRITICAL DATA
df = df.sort_values(['player', 'word', 'date']).dropna(subset=['success', 'complexity_score', 'demo.mpo'])
print(f'Pipeline integration complete: {len(df)} trials mapped.')

## 2. Training the Random Forest
We train the committee of decision trees to learn how Streaks and Efficiency predict future maintenance.

In [ ]:
features = ['s', 'consecutive_success', 'learning_velocity', 'rt_jitter', 'rt_efficiency', 'complexity_score', 'demo.mpo', 'demo.age', 'stars']
df_model = df.fillna(0)
train = df_model[df_model['phase'] != 'followup']
test = df_model[df_model['phase'] == 'followup']

X_train, y_train = train[features], train['success']
X_test, y_test = test[features], test['success']

# Train the model
model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
model.fit(X_train, y_train)
print('Random Forest model trained.')

## 3. Results: Relative Importance
Notice how the new features (Streaks, Efficiency) rank against clinical baselines.

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values()
plt.figure(figsize=(12, 8))
importances.plot(kind='barh', color=sns.color_palette('viridis', len(features)))
plt.title('Clinical Predictors of Maintenance (Success Streak & Efficiency included)')
plt.xlabel('Influence Score')
plt.show()

## 4. Evaluation: Performance Metrics

In [ ]:
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
accuracy = model.score(X_test, y_test)

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Score': [f'{accuracy:.4f}', f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}']
})
display(metrics_df)

# VISUALIZE THE RANDOM FOREST CONFUSION MATRIX (COUNTS & PERCENTAGES)
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Random Forest: Success Counts (Confusion Matrix)')
plt.xlabel('Predicted Outcome (1=Success)')
plt.ylabel('Actual Outcome (1=Success)')

plt.subplot(1, 2, 2)
cm_perc = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_perc, annot=True, fmt='.1%', cmap='Blues', cbar=False)
plt.title('Random Forest: Success Percentage (Normalized)')
plt.xlabel('Predicted Outcome (1=Success)')
plt.ylabel('Actual Outcome (1=Success)')
plt.tight_layout()
plt.show()